In [1]:
from pyspark.sql import SparkSession

jar_paths = [
    "/home/jovyan/work/jars/delta-spark_2.12-3.1.0.jar",
    "/home/jovyan/work/jars/delta-storage-3.1.0.jar",
    "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
]
jars_string = ",".join(jar_paths)

spark = SparkSession.builder \
    .appName("LakehouseSetup_Offline") \
    .config("spark.jars", jars_string) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("Connected successfully in Offline Mode! Ready to build the Lakehouse.")

Spark Version: 3.5.0
Connected successfully in Offline Mode! Ready to build the Lakehouse.


In [8]:
df_gold_customer_360 = spark.read.format("delta").load("s3a://olist-data/gold/customer_360")
df_gold_customer_360.orderBy(df_gold_customer_360["total_lifetime_value"].desc()).show(10)

+--------------------+--------------+------------+--------------------+-------------------+--------------------+--------------------+
|  customer_unique_id| customer_city|total_orders|total_lifetime_value| last_purchase_date|average_review_score|   favorite_category|
+--------------------+--------------+------------+--------------------+-------------------+--------------------+--------------------+
|0a0a92112bd4c708c...|rio de janeiro|           1|            13664.08|2017-09-29 15:24:52|                 1.0|      telefonia_fixa|
|da122df9eeddfedc1...|      araruama|           2|             7571.63|2017-04-01 15:58:41|                 5.0|     eletroportateis|
|763c8b1c9c68a0229...|    vila velha|           1|             7274.88|2018-07-15 14:49:44|                 1.0|      telefonia_fixa|
|dc4802a71eae9be1d...|  campo grande|           1|             6929.31|2017-02-12 20:37:36|                 5.0|utilidades_domest...|
|459bef486812aa252...|       vitoria|           1|            

In [9]:
df_gold_fact_orders_enriched = spark.read.format("delta").load("s3a://olist-data/gold/fact_orders_enriched")
df_gold_fact_orders_enriched.orderBy(df_gold_fact_orders_enriched["total_revenue"].desc()).show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+------------------+-----------+------------+--------------+--------------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|total_revenue|     total_freight|total_items|review_score| customer_city|customer_state|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+------------------+-----------+------------+--------------+--------------+
|1617b1357756262bf...|03caa2c082116e1d3...|   delivered|     2017-09-29 15:24:52|2017-10-02 15:28:20|         2017-10-10 15:43:17|          2017-10-17 18:22:29|   

In [11]:
df_gold_agg_sales = spark.read.format("delta").load("s3a://olist-data/gold/agg_sales_by_state_category_month")
df_gold_agg_sales.orderBy(df_gold_agg_sales["total_sales"].desc()).show(10)

+----------+-----------+--------------+---------------------+-----------+----------------+
|order_year|order_month|customer_state|product_category_name|total_sales|total_items_sold|
+----------+-----------+--------------+---------------------+-----------+----------------+
|      2018|          8|            SP|         beleza_saude|    49676.6|             409|
|      2018|          5|            SP|   relogios_presentes|   47480.02|             248|
|      2018|          5|            SP|         beleza_saude|    42537.5|             384|
|      2018|          6|            SP|         beleza_saude|    41407.7|             421|
|      2018|          2|            SP| informatica_acess...|   39849.49|             375|
|      2018|          4|            SP|         beleza_saude|   39665.18|             333|
|      2018|          5|            SP|      cama_mesa_banho|   38908.24|             411|
|      2018|          6|            SP|      cama_mesa_banho|   37740.78|             417|

In [12]:
df_gold_segment_rfm = spark.read.format("delta").load("s3a://olist-data/gold/segment_rfm")
df_gold_segment_rfm.orderBy(df_gold_segment_rfm["total_lifetime_value"].desc()).show(10)

+--------------------+--------------+------------+--------------------+-------------------+--------------------+--------------------+------------+-------+-------+-------+--------------------+
|  customer_unique_id| customer_city|total_orders|total_lifetime_value| last_purchase_date|average_review_score|   favorite_category|recency_days|R_Score|F_Score|M_Score|         RFM_Segment|
+--------------------+--------------+------------+--------------------+-------------------+--------------------+--------------------+------------+-------+-------+-------+--------------------+
|0a0a92112bd4c708c...|rio de janeiro|           1|            13664.08|2017-09-29 15:24:52|                 1.0|      telefonia_fixa|         367|      2|      2|      4|  2. Loyal Customers|
|da122df9eeddfedc1...|      araruama|           2|             7571.63|2017-04-01 15:58:41|                 5.0|     eletroportateis|         548|      1|      4|      4|          3. At Risk|
|763c8b1c9c68a0229...|    vila velha|   